# Argus — Dataset Creation (CNN+LSTM Windowed Image Sequences)

The fifth model family: a hybrid `TimeDistributed(CNN) → LSTM` that sees an actual windowed
*sequence* of face-crop images (1-6s, same multi-duration sliding-window scheme as
[`01_dataset_creation_lstm.ipynb`](./01_dataset_creation_lstm.ipynb)), rather than either a
single cropped image (`06`/`07`'s CNN) or hand-engineered geometric features (the LSTM). This is
the most complete fix for the single-instant-in-time limitation that capped every other
single-frame model around ~40%: it combines raw-pixel visual cues (which plain geometric ratios
can't see) with genuine temporal context (which a single image can't see either).

**Calibrated expectation, stated plainly up front:** this is also, by a wide margin, the most
data-hungry model built so far — a deep model over raw pixel *sequences*, trained on ~24
subjects. Real overfitting risk here is higher than anywhere else in this project, not lower.
Worth trying — it's the theoretically strongest architecture for this problem — but don't assume
"more complete architecture" automatically means "better result" on a dataset this size; treat it
as a real experiment with a real chance of underperforming the LSTM, not a foregone upgrade.

**Reads:** `dataset_processed/face_crops_index.csv` and the `.jpg` files it points to, written by
[`06_dataset_creation_face_crops.ipynb`](./06_dataset_creation_face_crops.ipynb) — **run that
first if you haven't** (it hasn't produced output yet as of this notebook being written). This
notebook does **not** re-extract or re-crop anything from video — it only builds a lightweight
window *index* (frame-range references into crops that already exist), the same principle that
turned the LSTM's dataset from thousands of `.npy` files into one CSV: reuse what's already
extracted, don't multiply storage by re-saving overlapping copies of it.

**Writes:** `dataset_processed/cnn_lstm_windows_index.csv` — one row per valid window, storing an
ordered, `;`-joined list of that window's real crop image paths (not the pixels themselves; those
are decoded lazily by `10_cnn_lstm_training.ipynb`'s `tf.data` pipeline, same lazy-loading pattern
`07_cnn_training.ipynb` already uses for single images).


## Google Drive Connection & Project Setup


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

project_folder = "/content/drive/MyDrive/Argus"
print(f"Google Drive successfully mounted! Base project directory: {project_folder}")


In [ ]:
import os

models_folder = f"{project_folder}/models"
dataset_folder = f"{project_folder}/dataset"
processed_folder = f"{dataset_folder}/dataset_processed"
face_crops_folder = f"{processed_folder}/face_crops"
face_crops_index_csv_path = os.path.join(processed_folder, "face_crops_index.csv")

if not os.path.exists(face_crops_index_csv_path):
    raise FileNotFoundError(
        f"'{face_crops_index_csv_path}' not found. Run 06_dataset_creation_face_crops.ipynb "
        "first -- this notebook only indexes windows over crops it produces, it doesn't extract "
        "any crops itself."
    )

print("Project folder structure validated.")


## Pipeline Configuration Constants

Same multi-duration window scheme as `01_dataset_creation_lstm.ipynb` (1.0-6.0s, 1s stride) --
deliberately kept identical so a window "means the same thing" across the geometric-feature LSTM
and this image-sequence model, and so the two can be fairly compared later.

**`MAX_TIMESTEPS_IMG = 60`, not 120.** The geometric LSTM's dataset pads to 120 (double the real
60-frame/6s max) for headroom, because each padded slot only costs 58 floats -- cheap regardless
of how much of it is padding. Here each padded slot is a full `(IMG_SIZE, IMG_SIZE, 3)` image
tensor -- padding to double the real max would double memory/compute for pure waste. Padding
exactly to the real max (60 frames = 6s @ 10 FPS) is the right tradeoff for images specifically;
this is a genuinely different choice from the geometric notebook's 120, not an oversight.


In [ ]:
sampling_fps = 10                     # Must match 06_dataset_creation_face_crops.ipynb
min_context_sec = 1
max_context_sec = 6.0
window_configs = [float(x) for x in range(int(min_context_sec), int(max_context_sec) + 1)]
stride_sec = 1.0

MAX_TIMESTEPS_IMG = int(max_context_sec * sampling_fps)  # 60 -- see note above on why not 120 here

# --- Raw level (1-6, from filenames) -> 3-class drowsiness label mapping ---
# Identical to every other dataset-creation notebook in this project.
EXTERNAL_SUBJECT_START = 7
RAW_LEVEL_TO_CLASS = {1: 1, 2: 1, 3: 2, 4: 2, 5: 3, 6: 3}  # 1=Alert, 2=Low Vigilant, 3=Drowsy

print(f"✅ Pipeline constants initialized. Windows: {window_configs}s, MAX_TIMESTEPS_IMG={MAX_TIMESTEPS_IMG}")


## Building the Window Index

For each clip `(subject, parent_video)`, slide the same 1-6s/1s-stride windows over its
**already-cropped** frames (loaded from `face_crops_index.csv`). Windowing is done over
`sample_idx` -- the consecutive sampled-frame position `06` now records -- **not** `frame_idx`
(the raw video frame count, which is spaced by that video's own `frame_stride` and would make
every window spuriously fail a naive contiguity check). A window is valid only if every
`sample_idx` in its range has a crop -- some are missing where the Face Detector found no
confident face, and exactly like the geometric LSTM discarding any window containing an invalid
frame, a single gap anywhere in the range invalidates the whole window rather than being skipped
over. This keeps every window a genuinely contiguous stretch of real time.


In [ ]:
import pandas as pd
import numpy as np

df_face_crops = pd.read_csv(face_crops_index_csv_path)
if 'sample_idx' not in df_face_crops.columns:
    raise ValueError(
        "face_crops_index.csv has no 'sample_idx' column -- re-run 06_dataset_creation_face_crops.ipynb "
        "(it was updated to record this alongside frame_idx; a CSV from before that update won't have it)."
    )
df_face_crops = df_face_crops.sort_values(['subject', 'parent_video', 'sample_idx']).reset_index(drop=True)

print(f"Loaded {len(df_face_crops)} indexed crops across "
      f"{df_face_crops.groupby(['subject', 'parent_video']).ngroups} clips.")


In [ ]:
def build_windows_for_clip(clip_df, window_configs, sampling_fps, stride_sec):
    """clip_df: rows for one (subject, parent_video), already sorted by sample_idx.

    Returns a list of dicts, one per valid window: the ordered image_path list for that window's
    real (contiguous, gap-free) frames -- no padding decided here, that happens at load time in
    10_cnn_lstm_training.ipynb, mirroring how 01_dataset_creation_lstm.ipynb keeps padding out of
    the generation step for everything except the final flattened output.
    """
    sample_idx_to_path = dict(zip(clip_df['sample_idx'], clip_df['image_path']))
    available_samples = sorted(sample_idx_to_path.keys())
    if not available_samples:
        return []

    windows = []
    for win_sec in window_configs:
        win_size = int(win_sec * sampling_fps)
        stride = int(stride_sec * sampling_fps)

        start_range_end = available_samples[-1] - win_size + 1
        for start in range(available_samples[0], start_range_end + 1, stride):
            end = start + win_size  # exclusive
            required = range(start, end)
            # Every sample_idx in [start, end) must have a crop -- any gap invalidates the window.
            if not all(si in sample_idx_to_path for si in required):
                continue
            image_paths = [sample_idx_to_path[si] for si in required]
            windows.append({
                'window_duration_sec': win_sec,
                'n_real_frames': win_size,
                'start_sample_idx': start,
                'end_sample_idx': end - 1,
                'image_paths': ';'.join(image_paths),
            })
    return windows


### Running the Window Builder Across All Clips


In [ ]:
from tqdm.auto import tqdm

rows = []
skipped = []

grouped = df_face_crops.groupby(['subject', 'parent_video'], sort=False)

for (subject, parent_video), clip_df in tqdm(grouped, desc="Building windows per clip"):
    level = clip_df['level'].iloc[0]  # constant per clip -- 06 already resolved subject-aware mapping

    clip_windows = build_windows_for_clip(clip_df, window_configs, sampling_fps, stride_sec)
    if not clip_windows:
        skipped.append((subject, parent_video, "No valid (gap-free) window of any configured duration"))
        continue

    for w in clip_windows:
        w['subject'] = subject
        w['parent_video'] = parent_video
        w['level'] = level
        rows.append(w)

print(f"\nBuilt {len(rows)} windows across {grouped.ngroups} clips.")
if skipped:
    print(f"Skipped {len(skipped)} clips with no valid window:")
    for subject, parent_video, reason in skipped[:10]:
        print(f"   - {subject}/{parent_video}: {reason}")
    if len(skipped) > 10:
        print(f"   ... and {len(skipped) - 10} more")


### Writing the Window Index CSV


In [ ]:
cnn_lstm_windows_csv_path = os.path.join(processed_folder, "cnn_lstm_windows_index.csv")

df_cnn_lstm_windows = pd.DataFrame(rows)
df_cnn_lstm_windows.to_csv(cnn_lstm_windows_csv_path, index=False)

print(f"✅ Window index written: {cnn_lstm_windows_csv_path}")
print(f"   Shape: {df_cnn_lstm_windows.shape}")


### Dataset Sanity Check

Verify class balance across windows, and preview one window's frame sequence to confirm the paths resolve to real, temporally-ordered images.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

df_cnn_lstm_windows = pd.read_csv(cnn_lstm_windows_csv_path)
level_counts = df_cnn_lstm_windows['level'].value_counts().sort_index()

print("--- Dataset Sanity Check ---")
print(f"Total windows: {len(df_cnn_lstm_windows)}")
print(f"Distinct subjects: {df_cnn_lstm_windows['subject'].nunique()}")
print(f"Window duration distribution:\n{df_cnn_lstm_windows['window_duration_sec'].value_counts().sort_index()}")
print("\nWindow counts per level:")
for lvl, count in level_counts.items():
    print(f"  Level {lvl}: {count} windows")

missing_levels = [l for l in [1, 2, 3] if l not in level_counts.index]
if not missing_levels:
    print("\n✅ Success: All 3 levels are present in the dataset.")
else:
    print(f"\n⚠️ Warning: Missing levels {missing_levels} in the processed data.")

# Preview one 6-second window's frame sequence to visually confirm temporal ordering.
sample = df_cnn_lstm_windows[df_cnn_lstm_windows['window_duration_sec'] == max_context_sec].sample(1, random_state=0).iloc[0]
sample_paths = sample['image_paths'].split(';')
print(f"\nPreviewing a {sample['window_duration_sec']}s window ({len(sample_paths)} frames), "
      f"subject={sample['subject']}, level={sample['level']}:")

n_preview = min(8, len(sample_paths))
preview_idx = np.linspace(0, len(sample_paths) - 1, n_preview, dtype=int)
fig, axes = plt.subplots(1, n_preview, figsize=(2 * n_preview, 2.5))
for ax, idx in zip(axes, preview_idx):
    ax.imshow(mpimg.imread(sample_paths[idx]))
    ax.set_title(f"t={idx}")
    ax.axis('off')
plt.tight_layout()
plt.show()
